# Phase 4 — Model Training & Evaluation

Train and evaluate baseline fraud-detection models using the leakage-safe
training/test datasets prepared in Phase 3.

**Models:** Logistic Regression, Random Forest, XGBoost  
**Metrics:** Precision, Recall, F1, ROC-AUC, PR-AUC  
**Primary metric:** PR-AUC due to extreme class imbalance.

In [1]:
import pandas as pd

X_train = pd.read_csv("creditCardFraud/data/processed/X_train_scaled.csv")

X_test = pd.read_csv("creditCardFraud/data/processed/X_test_scaled.csv")

y_train = pd.read_csv("creditCardFraud/data/processed/y_train.csv").squeeze("columns")

y_test = pd.read_csv("creditCardFraud/data/processed/y_test.csv").squeeze("columns")

print("X_train: ",X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)



X_train:  (226980, 30)
X_test : (56746, 30)
y_train: (226980,)
y_test : (56746,)


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import(
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train,y_train)

y_dummy = dummy_model.predict(X_test)

print("Precision: ", precision_score(y_test,y_dummy,zero_division=0))
print('Recall: ', recall_score(y_test,y_dummy,zero_division=0))
print("F1: ",f1_score(y_test,y_dummy,zero_division=0))
print("PR-AUC: ",average_precision_score(y_test,y_dummy))

Precision:  0.0
Recall:  0.0
F1:  0.0
PR-AUC:  0.0016741268107003137


In [4]:
from sklearn.linear_model import LogisticRegression


logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train,y_train)

y_pred_lr = logistic_model.predict(X_test)

y_prob_lr = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic Regression training complete. ")

Logistic Regression training complete. 


In [6]:
from sklearn.metrics import(
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

precision_lr = precision_score(y_test,y_pred_lr)
recall_lr = recall_score(y_test,y_pred_lr)
f1_lr = f1_score(y_test,y_pred_lr)
roc_auc_lr = roc_auc_score(y_test,y_prob_lr)
pr_auc_lr = average_precision_score(y_test,y_prob_lr)
cm_lr =  confusion_matrix(y_test,y_pred_lr)


print(f"Precision : {precision_lr:.4f}")
print(f"Recall    : {recall_lr:.4f}")
print(f"F1 Score  : {f1_lr:.4f}")
print(f"ROC-AUC   : {roc_auc_lr:.4f}")
print(f"PR-AUC    : {pr_auc_lr:.4f}")

print("\nConfusion Matrix:")
print(cm_lr)

Precision : 0.0564
Recall    : 0.8737
F1 Score  : 0.1060
ROC-AUC   : 0.9657
PR-AUC    : 0.6738

Confusion Matrix:
[[55263  1388]
 [   12    83]]


In [7]:
logistic_results = {
    "Model": "Logistic Regression",
    "Precision": precision_lr,
    "Recall": recall_lr,
    "F1": f1_lr,
    "ROC-AUC": roc_auc_lr,
    "PR-AUC": pr_auc_lr
}

print(logistic_results)

{'Model': 'Logistic Regression', 'Precision': 0.05642420122365738, 'Recall': 0.8736842105263158, 'F1': 0.10600255427841636, 'ROC-AUC': 0.9656530427762225, 'PR-AUC': 0.6738217282204635}


In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators = 200,
    class_weight = "balanced",
    random_state = 42,
    n_jobs = -1
)

rf_model.fit(X_train,y_train)

y_pred_rf = rf_model.predict(X_test)

y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("Random Forest training complete.")


Random Forest training complete.


In [9]:
precision_rf = precision_score(y_test,y_pred_rf)

recall_rf = recall_score(y_test,y_pred_rf)

f1_rf = f1_score(y_test,y_pred_rf)

roc_auc_rf = roc_auc_score(y_test,y_prob_rf)

pr_auc_rf = average_precision_score(y_test,y_prob_rf)

cm_rf = confusion_matrix(y_test,y_pred_rf)

print(f"Precision : {precision_rf:.4f}")
print(f"Recall    : {recall_rf:.4f}")
print(f"F1 Score  : {f1_rf:.4f}")
print(f"ROC-AUC   : {roc_auc_rf:.4f}")
print(f"PR-AUC    : {pr_auc_rf:.4f}")

print("\nConfusion Matrix:")
print(cm_rf)

Precision : 0.9714
Recall    : 0.7158
F1 Score  : 0.8242
ROC-AUC   : 0.9447
PR-AUC    : 0.8078

Confusion Matrix:
[[56649     2]
 [   27    68]]


In [10]:
# Store Random Forest metrics using the same structure as Logistic Regression.
# Keeping an identical structure makes model comparison straightforward later.

random_forest_results = {
    "Model": "Random Forest",
    "Precision": precision_rf,
    "Recall": recall_rf,
    "F1": f1_rf,
    "ROC-AUC": roc_auc_rf,
    "PR-AUC": pr_auc_rf
}

print(random_forest_results)

{'Model': 'Random Forest', 'Precision': 0.9714285714285714, 'Recall': 0.7157894736842105, 'F1': 0.8242424242424242, 'ROC-AUC': 0.9446849546949049, 'PR-AUC': 0.8077701218894578}


In [11]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators = 200,
    max_depth = 6,
    learning_rate = 0.1,
    subsample = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = scale_pos_weight,
    objective = "binary:logistic",
    eval_metric = "logloss",
    random_state = 42,
    n_jobs = -1
)

xgb_model.fit(X_train,y_train)

y_pred_xgb = xgb_model.predict(X_test)

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost trained succesfully.")

XGBoost trained succesfully.


## XGBoost Hyperparameters

- `n_estimators`: Number of boosting trees. More trees can improve learning,
  but too many can increase training time and overfitting.

- `max_depth`: Maximum depth of each tree. Larger values allow the model to
  learn more complex patterns, but can increase overfitting.

- `learning_rate`: Controls how much each new tree contributes. Smaller values
  usually require more trees but can produce better generalization.

- `subsample`: Fraction of training samples used to build each tree. Values
  below 1.0 introduce randomness and can reduce overfitting.

- `colsample_bytree`: Fraction of features considered by each tree. This adds
  additional randomness and can improve generalization.

- `min_child_weight`: Controls how much data is required for a tree split.
  Larger values make the model more conservative.

- `gamma`: Minimum loss reduction required before making a split. Larger values
  make the model more conservative.

### Tuning principle

We will not optimize for accuracy because the dataset is extremely imbalanced.
PR-AUC will be our primary tuning metric, with recall, precision, F1 and
ROC-AUC considered as secondary evaluation metrics.

In [12]:
precision_xgb = precision_score(y_test,y_pred_xgb)
recall_xgb = recall_score(y_test,y_pred_xgb)
f1_xgb = f1_score(y_test,y_pred_xgb)
roc_auc_xgb = roc_auc_score(y_test,y_prob_xgb)
pr_auc_xgb = average_precision_score(y_test,y_prob_xgb)
cm_xgb = confusion_matrix(y_test,y_pred_xgb)

print(f"Precision : {precision_xgb:.4f}")
print(f"Recall    : {recall_xgb:.4f}")
print(f"F1 Score  : {f1_xgb:.4f}")
print(f"ROC-AUC   : {roc_auc_xgb:.4f}")
print(f"PR-AUC    : {pr_auc_xgb:.4f}")

print("\nConfusion Matrix:")
print(cm_xgb)

Precision : 0.9367
Recall    : 0.7789
F1 Score  : 0.8506
ROC-AUC   : 0.9761
PR-AUC    : 0.8248

Confusion Matrix:
[[56646     5]
 [   21    74]]


In [12]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier

cv = StratifiedKFold(
    n_splits = 3,
    shuffle = True,
    random_state = 42
)

param_distributions = {
    "n_estimators": [100,200,300,500],
    "max_depth": [3,4,5,6,8],
    "learning_rate": [0.01,0.03,0.05,0.1,0.2],
    "subsample": [0.6,0.7,0.8,0.9,1.0],
    "colsample_bytree": [0.6,0.7,0.8,0.9,1.0],
    "min_child_weight": [1,3,5,10],
    "gamma": [0,0.1,0.3,0.5,1]
}

xgb_tuning_model = XGBClassifier(
    objective = "binary:logistic",
    eval_metric = "logloss",
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum(),
    random_state = 42,
    n_jobs = -1
)

xgb_random_search = RandomizedSearchCV(
    estimator = xgb_tuning_model,
    param_distributions = param_distributions,
    n_iter = 20,
    scoring = "average_precision",
    cv = cv,
    verbose = 1,
    random_state = 42,
    n_jobs = -1
)

xgb_random_search.fit(X_train,y_train)

print("XGBoost hyperparameter tuning complete.")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
XGBoost hyperparameter tuning complete.


In [13]:
print("Best PR-AUC: ",xgb_random_search.best_score_)
print("\nBest hyperparameters: ")
print(xgb_random_search.best_params_)

Best PR-AUC:  0.8492795335930431

Best hyperparameters: 
{'subsample': 0.7, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 0.6}


In [14]:
best_xgb_model = xgb_random_search.best_estimator_

y_pred_xgb_tuned = best_xgb_model.predict(X_test)

y_prob_xgb_tuned = best_xgb_model.predict_proba(X_test)[:, 1]


print("Tuned XGBoost predictions generated.")

Tuned XGBoost predictions generated.


In [15]:
precision_xgb_tuned = precision_score(y_test,y_pred_xgb_tuned)
recall_xgb_tuned = recall_score(y_test,y_pred_xgb_tuned)
f1_xgb_tuned = f1_score(y_test,y_pred_xgb_tuned)
roc_auc_xgb_tuned = roc_auc_score(y_test,y_prob_xgb_tuned)
pr_auc_xgb_tuned = average_precision_score(y_test,y_prob_xgb_tuned)
cm_xgb_tuned = confusion_matrix(y_test,y_pred_xgb_tuned)


print(f"Precision : {precision_xgb_tuned:.4f}")
print(f"Recall    : {recall_xgb_tuned:.4f}")
print(f"F1 Score  : {f1_xgb_tuned:.4f}")
print(f"ROC-AUC   : {roc_auc_xgb_tuned:.4f}")
print(f"PR-AUC    : {pr_auc_xgb_tuned:.4f}")

print("\nConfusion Matrix:")
print(cm_xgb_tuned)

Precision : 0.8941
Recall    : 0.8000
F1 Score  : 0.8444
ROC-AUC   : 0.9750
PR-AUC    : 0.8261

Confusion Matrix:
[[56642     9]
 [   19    76]]
